# Same-String Answerability Causal Replication v2

This notebook executes the preregistered fresh-unit v2 replication on free Colab. It reuses the proven v1 runtime while keeping a separate artifact namespace. Layer 18, multiplier 1.0, and the `user_prompt_end` anchor are locked and cannot be reselected.

The execution is pinned to commit `26188c9b9105d96446c0ea276fc84be5e444bd0e`, which contains the reviewed v2 source, config, tests, and hash-bound preregistration.

In [ ]:
from pathlib import Path
import os, subprocess, sys

PINNED_REPO_COMMIT = "26188c9b9105d96446c0ea276fc84be5e444bd0e"
assert len(PINNED_REPO_COMMIT) == 40 and all(
    character in "0123456789abcdef" for character in PINNED_REPO_COMMIT
), "PINNED_REPO_COMMIT must be a lowercase 40-character commit hash."

CHECKOUT = Path("/content/answerability-familiarity-v2")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "https://github.com/Fredo220/Answerability-x-Familarity-.git", str(CHECKOUT)], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "origin", PINNED_REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", PINNED_REPO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CHECKOUT / "requirements/fa-causal-colab.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))

if not os.environ.get("HF_TOKEN"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN through Colab Secrets before continuing."

In [ ]:
USE_DRIVE_CHECKPOINTS = True
ARTIFACT_ROOT = Path("/content/fa-causal-replication-v2")
if USE_DRIVE_CHECKPOINTS:
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=60_000)
        ARTIFACT_ROOT = Path("/content/drive/MyDrive/fa-causal-replication-v2")
    except (TimeoutError, ValueError) as error:
        print(f"Drive unavailable; using local checkpoints: {error}")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Artifact root: {ARTIFACT_ROOT}")

In [ ]:
import argparse, json, torch
from trajectory_extractor.fa_answerability_causal_cli import (
    CausalDependencies, CausalTokenizerBinding, HFCausalRunner,
    load_causal_config, prepare_causal, run_causal_validation,
    expected_causal_shards, run_causal_shard, evaluate_causal,
)

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
CONFIG = CHECKOUT / "configs/familiarity_answerability_causal_replication_v2.json"
V3_CORPUS = CHECKOUT / "release/familiarity_answerability/representation_replication_v3/same_string_replication_v3_manifest.json"
V3_ACTIVATIONS = CHECKOUT / "release/familiarity_answerability/representation_replication_v3/activations/activations-representation_train.manifest.json"
config = load_causal_config(CONFIG)
assert config.study_id == "same-string-answerability-causal-replication-v2"
assert config.validation_selection["mode"] == "locked_from_v1"
assert config.validation_selection["locked_layer"] == 18
assert config.validation_selection["locked_multiplier"] == 1.0

In [ ]:
# Preparation constructs and audits the fresh v2 corpus before model loading.
prepared = prepare_causal(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    v3_corpus_manifest=str(V3_CORPUS),
    v3_training_activation_manifest=str(V3_ACTIVATIONS),
    output_dir="prepared",
))
print(json.dumps(prepared, indent=2))

In [ ]:
# Load Gemma once. Validation and all 432 shards reuse this exact instance.
runner = HFCausalRunner.from_pretrained(config)
binding = CausalTokenizerBinding(
    tokenizer=runner.tokenizer, model_id=config.model_id,
    model_revision=config.model_revision, tokenizer_id=config.model_id,
    tokenizer_revision=config.tokenizer_revision,
    chat_template_sha256=config.chat_template_sha256,
)
deps = CausalDependencies(
    tokenizer_loader=lambda _config: binding,
    runner_factory=lambda _config: runner,
)

In [ ]:
# Locked validation checks runtime, format, and preservation without reselection.
validation = run_causal_validation(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    prepare_manifest=prepared["prepare_manifest"],
    output_dir="validation", resume=True,
), dependencies=deps)
selection = json.loads(Path(validation["selection_manifest"]).read_text())
assert selection["layer_id"] == 18
assert selection["multiplier"] == 1.0
print(json.dumps(validation, indent=2))

In [ ]:
# Each receipt is atomic. Rerun this cell after interruption to resume.
seal = json.loads(Path(validation["seal_manifest"]).read_text())
schedule = expected_causal_shards(seal)
assert len(schedule) == 432, f"Expected 432 sealed shards, found {len(schedule)}"
for index, item in enumerate(schedule, start=1):
    shard_result = run_causal_shard(argparse.Namespace(
        config=str(CONFIG), root=str(ARTIFACT_ROOT),
        prepare_manifest=prepared["prepare_manifest"],
        seal_manifest=validation["seal_manifest"],
        split=item["split"], control=item["control"],
        unit_id=item["unit_id"], member=item["member"],
        output_dir="evidence", resume=True,
    ), dependencies=deps)
    if index % 12 == 0 or index == len(schedule):
        print(f"{index}/{len(schedule)}: {shard_result['status']}")

In [ ]:
# One-use protected evaluation. Run only after all 432 receipts exist.
final_result = evaluate_causal(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    prepare_manifest=prepared["prepare_manifest"],
    seal_manifest=validation["seal_manifest"],
    evidence_dir="evidence", output_dir="results",
), dependencies=deps)
print(json.dumps(final_result, indent=2))

In [ ]:
# Export the complete artifact tree after successful final evaluation.
import shutil
from google.colab import files

ZIP_BASE = Path("/content/fa-causal-replication-v2-artifacts")
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=ARTIFACT_ROOT)
print(f"Created {zip_path}")
files.download(zip_path)